In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
housing_df = pd.read_csv('housing_train.csv')

In [ ]:
print(housing_df.info())
housing_df.head(2)

In [4]:
dataset_cols = ['region', 'type', 'sqfeet', 'beds', 'baths',
    'comes_furnished', 'laundry_options', 'parking_options',
    'cats_allowed', 'dogs_allowed','price']
#we will be predicting the price of the house the above mentioned features
housing_df = housing_df[dataset_cols]

In [ ]:
housing_df.head(2)

In [ ]:
print(housing_df.isnull().sum())
housing_df.shape


In [ ]:
housing_df['laundry_options'].value_counts()

In [ ]:
housing_df['parking_options'].value_counts()

In [ ]:
housing_df.drop_duplicates(inplace=True)
housing_df.shape

In [ ]:
housing_df.dropna(inplace=True)
housing_df.shape

In [ ]:
housing_df[dataset_cols].describe() #Each columns contains outlier - we need to remove them


In [12]:
# 1. Remove rows with 0 or too large price
housing_df = housing_df[(housing_df['price'] >= 300) & (housing_df['price'] <= 10000)]

# 2. Remove rows with 0 or overly large sqfeet
housing_df = housing_df[(housing_df['sqfeet'] >= 150) & (housing_df['sqfeet'] <= 5000)]

# 3. Remove rows with 0 or absurd number of beds (like 1100)
housing_df = housing_df[(housing_df['beds'] >= 1) & (housing_df['beds'] <= 10)]

# 4. Remove rows with 0 or too many baths (like 75)
housing_df = housing_df[(housing_df['baths'] >= 1) & (housing_df['baths'] <= 10)]

#fixing comes_furnished
housing_df = housing_df[(housing_df['comes_furnished'] == 1) | (housing_df['comes_furnished'] == 0)]

#fixing cats_allowed and dogs_allowed
housing_df = housing_df[(housing_df['cats_allowed'] == 1.0) | (housing_df['cats_allowed'] == 0.0)]
housing_df = housing_df[(housing_df['dogs_allowed'] == 1) | (housing_df['dogs_allowed'] == 0)]

In [13]:
housing_df[dataset_cols].describe()
housing_df_sample = housing_df 
housing_df_sample.to_csv('cleaned_housing_df.csv', index=False)









In [ ]:
sns.histplot(housing_df['price'], bins=50, kde=True)

In [ ]:
features = ['region', 'type', 'sqfeet', 'beds', 'baths',
    'comes_furnished', 'laundry_options', 'parking_options',
    'cats_allowed', 'dogs_allowed']
target = 'price'
#On the first run I got abnormal result - the graphs dont make any sense this means that the data is not distributed normally and contains outliers
for feature in features:
    plt.scatter(x=feature, y=target, data=housing_df)
    plt.xlabel(feature)
    plt.ylabel('Price')
    plt.title(f'{feature} vs Price')
    plt.show()

In [ ]:
housing_df.head()

In [17]:
#Converting String data into numerical Data using cat codes
housing_df['region'] = housing_df['region'].astype('category').cat.codes
housing_df['type'] = housing_df['type'].astype('category').cat.codes
housing_df['comes_furnished'] = housing_df['comes_furnished'].astype('category').cat.codes
housing_df['laundry_options'] = housing_df['laundry_options'].astype('category').cat.codes
housing_df['parking_options'] = housing_df['parking_options'].astype('category').cat.codes

In [ ]:
housing_df.head()
#Data is ready to be used for training

In [19]:
def get_xy(df):
    return df[features],df[target]

In [20]:
from sklearn.model_selection import train_test_split

In [21]:
#Spliting the data into training and testing sets
train , test = train_test_split(housing_df, test_size=0.2, random_state=42)

In [22]:
train_x, train_y = get_xy(train)
test_x, test_y = get_xy(test)

In [23]:
from sklearn.ensemble import RandomForestRegressor

In [24]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42) #Random Forest Regressor model where the hyper parameter n_estimators is the number of trees in the forest and random_state=42 is used for reproducibility - that everytime i will get the same result using the model

In [ ]:
rf_model.fit(train_x, train_y)
rf_model.score(test_x, test_y)

In [26]:
y_pred = rf_model.predict(test_x)

In [ ]:
encoded_sample = np.array([[57, 1, 950, 2, 1, 1, 1, 0, 3, 2]])
rf_model.predict(encoded_sample)

In [28]:
#So far from random forest model we have a score of 0.64 which is pretty decent but not so good so we will use a different model - XGBoost
from xgboost import XGBRegressor

In [29]:
xgb_model = XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42)

In [ ]:
xgb_model.fit(train_x, train_y)
xgb_model.score(test_x, test_y)

In [33]:
def predict_price(features):
    return rf_model.predict(features)
#The input is a numpy array of features 

In [ ]:
housing_df.info()

In [ ]:
input_array = np.array([[21, 0, 1009, 2, 1.0, 0, 1, 4, 1, 1]])
predicted_price = predict_price(input_array)
print(predicted_price)  

In [41]:
#Saving our model
import joblib

In [42]:
joblib.dump(rf_model, 'model.pkl')

['model.pkl']

In [46]:
loaded_model = joblib.load("model.pkl")

# Run a quick prediction to confirm
sample_input = [[0, 1, 950, 2, 1, 1, 0, 2, 1, 1]]  # Example input
print(loaded_model.predict(sample_input))

[2858.14]


c:\Users\majid\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
